# KV Cache Benchmark Colab Demo

Use this notebook on a Google Colab GPU runtime. It installs dependencies, checks CUDA, runs smoke and full benchmarks, plots results, and zips outputs.

## 1. Open Or Clone The Repo

If you uploaded/opened this notebook from inside the repo, skip the clone cell. Otherwise set `REPO_URL` and run it.

In [ ]:
import os, pathlib
REPO_URL = ''  # optional: 'https://github.com/<you>/<repo>.git'
if REPO_URL and not pathlib.Path('kvcache').exists():
    !git clone $REPO_URL kvcache
if pathlib.Path('kvcache').exists():
    os.chdir('kvcache')
print('Working directory:', os.getcwd())
assert pathlib.Path('benchmarks/benchmark.py').exists(), 'Run this from the repo root or set REPO_URL.'

## 2. Install Dependencies

In [ ]:
!pip install -r requirements-colab.txt

## 3. Check GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 4. Smoke Benchmark

In [ ]:
!bash scripts/run_colab_smoke.sh

## 5. Full Benchmark

The default script uses Qwen2.5-3B. If Colab runs out of memory, switch to Qwen2.5-1.5B by uncommenting the second command.

In [ ]:
!bash scripts/run_colab_full.sh
# !MODEL_NAME=Qwen/Qwen2.5-1.5B bash scripts/run_colab_full.sh

## 6. Plot Results

In [ ]:
!python benchmarks/plot_results.py --summary results/colab_full/summary_runs.csv --output-dir results/colab_full/plots

## 7. Inspect Summary

In [ ]:
import pandas as pd
df = pd.read_csv('results/colab_full/summary_runs.csv')
cols = ['mode','sequence_length','tokens_per_sec','latency_per_token_ms','kv_memory_mb','compression_ratio','accuracy_cosine','kl_divergence','next_token_agreement']
df[cols].round(4)

## 8. Zip Results For Download

In [ ]:
!zip -r kv_cache_results.zip results/colab_smoke results/colab_full
from google.colab import files
files.download('kv_cache_results.zip')